# 00 — Colab Setup

Run this notebook **first**, once per Colab runtime. It:
1. Clones this repo (or pulls latest if already cloned)
2. Installs `requirements.txt`
3. Downloads PTB-XL (public, no login) — optionally caches it to your
   Google Drive so you don't re-download every session
4. Runs the unit tests as a sanity check

Every other notebook in `notebooks/` assumes this has been run in the
current runtime.

## 1. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/<your-username>/mm-dg-healthcare.git"  # TODO: replace with your repo URL
REPO_DIR = "/content/mm-dg-healthcare"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already present, pulling latest...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 2. Install dependencies

Colab already has `torch` preinstalled and compatible with GPU runtimes —
we skip reinstalling it here to avoid a slow, unnecessary CUDA-version
mismatch reinstall. Everything else in `requirements.txt` installs
normally.

In [ ]:
!pip install -q pandas scikit-learn scipy PyYAML tqdm wfdb transformers pytest
print("Dependencies installed.")

## 3. (Optional but recommended) Mount Google Drive for data caching

PTB-XL is ~1.7 GB. Downloading it every fresh Colab session is slow.
Caching it on Drive means you download once and every future session just
copies from Drive instead. Skip this cell if you'd rather download fresh
each time (e.g. on a shared/temporary account).

In [ ]:
USE_DRIVE_CACHE = True  # set False to skip Drive entirely

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = "/content/drive/MyDrive/mm-dg-healthcare-data/ptbxl"
    import os
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    print(f"Drive cache dir: {DRIVE_CACHE_DIR}")

## 4. Download PTB-XL (skips automatically if already present)

In [ ]:
import os
os.makedirs("data/raw/ptbxl", exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"

!wget -q --show-progress -N -c -np -nH --cut-dirs=3 -P data/raw/ptbxl {BASE}/ptbxl_database.csv
!wget -q --show-progress -N -c -np -nH --cut-dirs=3 -P data/raw/ptbxl {BASE}/scp_statements.csv
!wget -r -N -c -np -nH --cut-dirs=3 -P data/raw/ptbxl {BASE}/records100/

!ls data/raw/ptbxl
!du -sh data/raw/ptbxl

Streaming output truncated to the last 5000 lines.
Saving to: ‘data/raw/ptbxl/records100/00000/00053_lr.hea’

records100/00000/00 100%[===================>]     599  --.-KB/s    in 0s      

2026-10-13 03:21:34 (69.6 MB/s) - ‘data/raw/ptbxl/records100/00000/00053_lr.hea’ saved [599/599]

--2026-10-13 03:21:34--  https://physionet.org/files/ptb-xl/1.0.3/records100/00000/00054_lr.dat
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 24000 (23K) [application/octet-stream]
Saving to: ‘data/raw/ptbxl/records100/00000/00054_lr.dat’

records100/00000/00 100%[===================>]  23.44K   129KB/s    in 0.2s    

2026-10-13 03:21:35 (129 KB/s) - ‘data/raw/ptbxl/records100/00000/00054_lr.dat’ saved [24000/24000]

--2026-10-13 03:21:35--  https://physionet.org/files/ptb-xl/1.0.3/records100/00000/00054_lr.hea
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 595 [text/plain]
Saving to: ‘

## 5. Sanity check: run the unit tests (no dataset needed for these)

In [ ]:
!python -m pytest tests/ -v

## 6. Quick data sanity check

Confirms the metadata loads and the domain-shift `site` column looks as
expected before moving on to `01_data_exploration_ptbxl.ipynb`.

In [ ]:
from src.data.ptbxl_dataset import load_ptbxl_metadata

metadata = load_ptbxl_metadata("data/raw/ptbxl")
print(f"Loaded {len(metadata)} records")
print("\nRecords per site:")
print(metadata["site"].value_counts().sort_index())
print("\nRecords per diagnostic superclass:")
print(metadata["superclass"].value_counts())

Setup complete. Continue to `01_data_exploration_ptbxl.ipynb`.